# MOKE Testing Notebook

Run these cells from top to bottom. The default path uses the normal `MokeMeasurement` with virtual instruments and the separate magnetic-material model. Replace the instrument setup and illustrative calibration before using real hardware.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from piec.analysis.field_calibration import FieldCalibration
from piec.drivers.dmm.virtual_dmm import VirtualDMM
from piec.drivers.sourcemeter.virtual_sourcemeter import VirtualSourcemeter
from piec.measurement.moke import MokeMeasurement
from piec.simulation.hysteretic_magnetic_material import HystereticMagneticMaterial

## 2. Load the source-output-to-field calibration

In [ ]:
calibration_path = Path("example_calibration.csv")
if not calibration_path.exists():
    calibration_path = Path("Measurements/MOKE/example_calibration.csv")

calibration = FieldCalibration.load_csv(calibration_path)
print(calibration.to_dict())

## 3. Virtual instrument setup

The DMM receives a simulated detector voltage. The virtual instruments remain generic; the optical response and hysteresis are wired together here as setup configuration.

In [ ]:
source = VirtualSourcemeter("VIRTUAL")
detector_dmm = VirtualDMM("VIRTUAL")
material = HystereticMagneticMaterial(coercive_field=50, switching_width=10)
rng = np.random.default_rng(0)

def read_detector_voltage():
    output = source.state["source_voltage"]
    field = calibration.field_at_output(output)
    magnetization = material.apply_field(field)
    return 0.02 * magnetization + rng.normal(0.0, 0.0002)

detector_dmm.set_voltage_reader(read_detector_voltage)

## 4. Define one closed source cycle

In [ ]:
output_min, output_max = calibration.output_range
outputs = np.r_[
    np.linspace(output_min, output_max, 101),
    np.linspace(output_max, output_min, 101)[1:],
]
assert outputs[0] == outputs[-1]

## 5. Construct and run the ordinary MOKE measurement

In [ ]:
experiment = MokeMeasurement(
    sourcemeter=source,
    dmm=detector_dmm,
    calibration=calibration,
    output_values=outputs,
    compliance=0.01,
    max_output_step=0.25,
    dwell_time=0.0,
    ramp_delay=0.0,
    n_cycles=3,
    average_cycles=3,
    geometry="in-plane",
    save_dir=str(Path.cwd()),
)
data = experiment.run_experiment(save=False)

## 6. Inspect raw, last-cycle, and cycle-average data

In [ ]:
snapshot = experiment.snapshot()
display(snapshot.raw.head())
display(snapshot.last_cycle.head())
display(snapshot.cycle_average.head())

In [ ]:
fig, ax = plt.subplots()
ax.plot(snapshot.raw[snapshot.field_column], snapshot.raw["detector_voltage (V)"], color="0.7", label="raw")
ax.plot(snapshot.last_cycle[snapshot.field_column], snapshot.last_cycle["detector_voltage (V)"], label="last cycle")
ax.plot(snapshot.cycle_average[snapshot.field_column], snapshot.cycle_average["detector_voltage (V)"], linewidth=3, label="cycle average")
ax.set(xlabel=snapshot.field_column, ylabel="detector_voltage (V)", title="Virtual MOKE loop")
ax.legend();

## 7. Real hardware template

Use a measured calibration file and supported driver addresses. The numerical safety settings below are placeholders, not recommendations for your magnet. Keep this cell commented until the complete setup has been checked.

In [ ]:
# from piec.drivers.autodetect import autodetect
# from piec.drivers.dmm.dmm import DMM
# from piec.drivers.sourcemeter.sourcemeter import Sourcemeter
#
# source = autodetect(address="GPIB0::24::INSTR", required_type=Sourcemeter)
# detector_dmm = autodetect(address="GPIB0::16::INSTR", required_type=DMM)
# calibration = FieldCalibration.load_csv("in_plane_calibration.csv")
# outputs = np.r_[np.linspace(-5, 5, 101), np.linspace(5, -5, 101)[1:]]
# experiment = MokeMeasurement(
#     sourcemeter=source, dmm=detector_dmm, calibration=calibration,
#     output_values=outputs, compliance=0.01, max_output_step=0.1,
#     dwell_time=0.1, ramp_delay=0.01, n_cycles=3,
#     geometry="in-plane", save_dir="results",
# )
# data = experiment.run_experiment()

## 8. Optional measured-field reader

For a gaussmeter analog output read by another DMM, provide a callable that returns field in the same units as the calibration:

```python
def read_field_in_oe():
    return field_dmm.get_voltage() * oe_per_volt + field_offset

# Add to MokeMeasurement(...):
# field_reader=read_field_in_oe,
# field_reader_unit=calibration.field_unit,
# field_reader_name=field_dmm.idn(),
```